# 🚀 Fine-tuning CodeLlama with QLoRA on Google Colab

This notebook demonstrates how to fine-tune **CodeLlama-7b-Instruct** using **QLoRA (4-bit quantization)** to create a FastAPI specialist model. The notebook is specifically designed to run in Google Colab's environment.

## 🖥️ Hardware Requirements

This notebook requires:
- GPU: Google Colab's T4 or A100
- VRAM: ~15GB
- Training Time: ~1-2 hours

## 🔑 Prerequisites

1. A HuggingFace account and access token (for saving model)
2. ~15GB free GPU memory
3. Stable internet connection

## 📋 What This Notebook Does

1. **Setup**: Install dependencies and prepare environment
2. **Dataset**: Load FastAPI training examples
3. **Fine-tuning**: Train model using QLoRA
4. **Save**: Push model to HuggingFace Hub
5. **Test**: Run inference on fine-tuned model
6. **Evaluate**: Compare with base model

## ⚠️ Important Notes

- All work is done in Colab's temporary environment
- Model is saved to HuggingFace Hub for persistence
- No local file system dependencies


In [ ]:
# Install dependencies
%pip install -q transformers==4.35.0 datasets==2.14.5 peft==0.6.0 accelerate==0.24.0 bitsandbytes==0.39.1 huggingface_hub
%pip install -q torch==2.2.0+cu118 torchvision==0.17.0+cu118 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118

# Import required libraries
import os
import sys
import torch
from huggingface_hub import login
from datetime import datetime

# Verify CUDA is available
print('🔍 Checking GPU...')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    
# Login to HuggingFace Hub
print('\n🔐 Please enter your HuggingFace token when prompted...')
login()  # This will prompt for your token

# Clone repository
print('\n📦 Cloning repository...')
!git clone https://github.com/your-username/Fine-Tuning-Open-Source-LLM.git
%cd Fine-Tuning-Open-Source-LLM

# Add project root to Python path
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import our modules
from model.load_base_model import ModelLoader
from data.prepare_dataset import DatasetPreparator
from train.run_lora_finetune import run_training
from evaluate.fastapi_evaluator import FastAPIEvaluator

print('\n✅ Setup completed successfully!')


In [ ]:
# Load model with QLoRA configuration
print("Loading CodeLlama-7b-Instruct with QLoRA configuration...")
model_loader = ModelLoader("configs/lora_config.json")

print("Loading base model with 4-bit quantization and preparing for LoRA...")
model, tokenizer = model_loader.load_model_and_tokenizer()

# Check model memory usage
if torch.cuda.is_available():
    print(f"GPU Memory Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")


In [ ]:
# Set up model ID for HuggingFace Hub
username = input("Enter your HuggingFace username: ")
model_name = "fastapi-specialist-codellama-7b"
hub_model_id = f"{username}/{model_name}"

print(f"\nModel will be saved as: {hub_model_id}")

# Prepare dataset
print("\nPreparing dataset...")
dataset_preparator = DatasetPreparator(tokenizer=tokenizer)
dataset = dataset_preparator.prepare_dataset()
print(f"Dataset prepared with {len(dataset)} examples")

# Start fine-tuning
print("\nStarting fine-tuning process...")
print("This will take 1-2 hours. The model will be automatically saved to HuggingFace Hub.")

trained_model = run_training(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    output_dir="/content/temp_checkpoints",  # Temporary Colab directory
    push_to_hub=True,
    hub_model_id=hub_model_id
)

print("\n✨ Training completed!")
print(f"Model saved to HuggingFace Hub: {hub_model_id}")

# Save configuration for future reference
model_config = {
    "base_model": "codellama/CodeLlama-7b-Instruct-hf",
    "fine_tuned_model": hub_model_id,
    "training_time": datetime.now().isoformat(),
    "dataset_size": len(dataset)
}

print("\nModel configuration:")
for key, value in model_config.items():
    print(f"- {key}: {value}")


In [ ]:
# Define test prompts
test_prompts = [
    {
        "category": "Basic Endpoint",
        "prompt": "Create a FastAPI GET endpoint that returns a list of users with pagination support."
    },
    {
        "category": "Data Validation",
        "prompt": "Create a FastAPI POST endpoint for user registration with email validation using Pydantic."
    },
    {
        "category": "Authentication",
        "prompt": "Create a FastAPI endpoint that requires JWT authentication and returns user profile data."
    },
    {
        "category": "Database",
        "prompt": "Create a FastAPI endpoint that uses SQLAlchemy to fetch and return a list of blog posts."
    }
]

def format_prompt(instruction):
    """Format prompt for CodeLlama."""
    return f"[INST] {instruction} [/INST]"

def generate_response(prompt, model, tokenizer, max_new_tokens=512):
    """Generate response from the model."""
    inputs = tokenizer(format_prompt(prompt), return_tensors="pt", padding=True)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
        
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from response
    response = response[len(format_prompt(prompt)):].strip()
    return response

print("🧪 Testing model on FastAPI prompts...")
print("="*80)

for test in test_prompts:
    print(f"\n📝 Category: {test['category']}")
    print(f"Prompt: {test['prompt']}\n")
    
    response = generate_response(test['prompt'], model, tokenizer)
    print("Generated Code:")
    print("-"*40)
    print(response)
    print("="*80)

print("\n✅ Testing completed!")


In [ ]:
from evaluate.fastapi_evaluator import FastAPIEvaluator
import json
from datetime import datetime

print("🚀 Initializing FastAPI evaluator...")
evaluator = FastAPIEvaluator()

# First evaluate base model
print("\n📊 Evaluating base CodeLlama model...")
base_results = evaluator.evaluate_model(
    model_name="codellama/CodeLlama-7b-Instruct-hf",
    temperature=0.1
)

# Then evaluate fine-tuned model
print("\n📊 Evaluating fine-tuned FastAPI specialist...")
specialist_results = evaluator.evaluate_model(
    model_name="codellama/CodeLlama-7b-Instruct-hf",
    adapter_path="outputs/checkpoints",
    temperature=0.1
)

# Compare results
print("\n📈 Comparing model performance...")
comparison = evaluator.compare_models(base_results, specialist_results)

# Save evaluation results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_file = f"outputs/evaluation/model_comparison_{timestamp}.json"

evaluation_results = {
    "base_model": base_results,
    "specialist_model": specialist_results,
    "comparison": comparison,
    "timestamp": timestamp
}

with open(results_file, "w") as f:
    json.dump(evaluation_results, f, indent=2)

print(f"\n✅ Evaluation results saved to {results_file}")

# Print summary
print("\n📋 Evaluation Summary")
print("="*80)
print("\nBase Model Performance:")
evaluator.print_evaluation_summary(base_results)
print("\nSpecialist Model Performance:")
evaluator.print_evaluation_summary(specialist_results)
print("\nImprovement Analysis:")
print(f"- Syntax Correctness: {comparison['syntax_improvement']}%")
print(f"- Pattern Matching: {comparison['pattern_improvement']}%")
print(f"- Functional Tests: {comparison['functional_improvement']}%")
print(f"- Overall Score: {comparison['overall_improvement']}%")


In [ ]:
# Complex FastAPI task combining multiple aspects
complex_prompt = """Create a FastAPI endpoint for a blog post system that:
1. Uses proper dependency injection
2. Implements JWT authentication
3. Uses Pydantic models for request/response
4. Includes pagination and filtering
5. Uses SQLAlchemy for database operations
6. Implements proper error handling
7. Follows FastAPI best practices"""

print("🚀 Generating solution for complex FastAPI task...")
print("="*80)
print("Prompt:", complex_prompt)
print("\nGenerated Solution:")
print("-"*40)

response = generate_response(complex_prompt, model, tokenizer, max_new_tokens=1024)
print(response)

print("\n✨ You can now use this code in your FastAPI application!")
print("Remember to:")
print("1. Install required dependencies (fastapi, sqlalchemy, python-jose, etc.)")
print("2. Set up your database connection")
print("3. Configure your JWT secret key")
print("4. Add proper documentation using FastAPI's automatic OpenAPI generation")
